# Aspect-Based Sentiment Analysis with Ollama

In the previous notebook, we completed the task of creating "Ground Truths": human-verified labels for our reviews. Now, we're going to test a machine to see if it can match our human judgement.

We aren't just chatting with the model; we're using it as a processing engine to turn unstructured text into structured data.

---
## 1.&nbsp; Set-up 🛠️

First, we need to import the tools that allow Python to talk to the Ollama application running in the background.

**Note:** You must have the Ollama app running on your computer, and you must have pulled the model by running `ollama run llama3.2` in your terminal before running this code.

In [1]:
import ollama
import pandas as pd
import time
from pydantic import BaseModel, Field
import json

Let's run a quick connection test to ensure Python can speak to the model.

In [2]:
try:
    response = ollama.chat(
        model='llama3.2',
        messages=[
            {
                'role': 'user',
                'content': 'Are you ready to analyse some data? Answer yes or no.'
            }
        ]
    )
    print("Model Response:", response['message']['content'])
except Exception as e:
    print("Error:", e)
    print("⚠️ Make sure the Ollama app is running and you have downloaded llama3.2!")

Model Response: Yes.


If you see "Model Response: Yes", we're good to go. Here's what we just did:

* **`ollama.chat`**: This is the function that sends our message to the local server.
* **`model='llama3.2'`**: We specify exactly which brain we want to use.
* **`role`**: We use `'user'` for our inputs and `'assistant'` for the model's replies. This history helps the model understand context.
* **`content`**: The actual string of text we want the model to read.

**Why `ollama.chat`?**

You might be wondering: *"If we just want to process individual reviews and output JSON, why are we using a `chat` function? Doesn't that imply a continuous conversation with memory?"*

It's a great question. While Ollama does have a memoryless `ollama.generate` function, we are intentionally using `ollama.chat` for two reasons:

1. **It's Stateless by Default:** The `ollama.chat` function only "remembers" what is inside the `messages` list you pass it. Because we will be creating a brand new `messages` list for every single review in our dataset, the model forgets everything between loops. We pay zero "memory tax" for using it.
2. **Built-in Formatting:** Modern models like Llama 3 are "Instruction Tuned". They expect a strict separation between System Instructions (the rules) and User Prompts (the data). Under the hood, they use hidden control tokens to separate these. When we use `ollama.chat`, the Python library automatically handles this complex formatting for us, leading to much more reliable JSON outputs than if we glued the prompt and review together into a single string.

### 1.2 Enforcing Structure with Pydantic

We can strictly enforce the JSON output using a **Pydantic Schema**. Pydantic allows us to define the exact shape of the data we want (including data types like floats and strings).

By passing this schema directly to Ollama, we guarantee the model's output will match our expected format, eliminating the need to guess if the JSON is valid.

In [3]:
# Define the structure for a single aspect
class AspectScore(BaseModel):
    score: float | None = None
    quote: str | None = None

# Define the overall structure we expect the model to return
class ReviewAnalysis(BaseModel):
    gameplay: AspectScore = AspectScore()
    graphics: AspectScore = AspectScore()
    story: AspectScore = AspectScore()
    performance: AspectScore = AspectScore()
    overall: AspectScore = AspectScore()

LLMs are unpredictable. Even with strict JSON mode, a model might completely drop a key instead of returning "score": null.

By setting defaults (`= None` or `= AspectScore()`) we guarantee if the Ollama model drops the key, Pydantic will calmly step in, assign the default value of None, and let your code keep running. When dealing with LLMs, defensive programming is a must.

---
## 2.&nbsp; Prompt Engineering 📝

If we just ask the model "How's the game?", it might write us a poem, a long essay, or a vague summary. We're Data Analysts, not poets. Our data needs **structure** so we can put it into a spreadsheet.

To achieve this, we use a **System Prompt**. This acts as the "rules of the game" for the model. We're going to explicitly tell the model that it's an expert critic and, crucially, we'll force it to output data in a specific JSON format.


In [4]:
# We dynamically inject the schema into the prompt
schema_instructions = ReviewAnalysis.model_json_schema()

SYSTEM_PROMPT = f"""
You are an expert Video Game Critic.
Your task is to analyse game reviews and extract sentiment scores for specific aspects.

For each category (Gameplay, Graphics, Story, Performance, Overall), assign a score:
-1.0 (Strong Negative)
-0.5 (Negative)
 0.0 (Neutral / Mixed)
+0.5 (Positive)
+1.0 (Strong Positive)
null (Not Mentioned)

Also extract a short quote as evidence. If the aspect is not mentioned, the score and the quote should be null.

Output ONLY valid JSON that strictly matches this schema:
{json.dumps(schema_instructions, indent=2)}
"""

---
## 3.&nbsp; Analysing a Single Review 🧪

Before we run this on thousands of rows, we must test it on a single example to ensure our prompt works. We'll use a manually written review that is deliberately complex - it praises the graphics but hates the combat.

In [5]:
test_review = "The world design is a masterpiece, absolutely stunning to look at. The story is enjoyable enough, if a bit simple. Unfortunately, the combat is totally broken and unresponsive, making it a nightmare to play."

### 3.1. Setting the Temperature
In the code below, you'll notice we set `options={'temperature': 0}`.

* **High Temperature (e.g., 0.8):** Good for creative writing. The model takes risks and is inventive.
* **Low Temperature (e.g., 0):** Good for data extraction. The model becomes deterministic, boring, and predictable. This reduces the chance of it "hallucinating" random facts.

In [6]:
response = ollama.chat(
    model='llama3.2',
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': test_review}
    ],
    # This passes the Pydantic schema to Ollama to enforce structure
    format=ReviewAnalysis.model_json_schema(),
    options={'temperature': 0}
)

Let's look at the raw object that Ollama returned. It contains a lot of technical metadata.

In [7]:
response


ChatResponse(model='llama3.2', created_at='2026-08-06T19:15:04.100022Z', done=True, done_reason='stop', total_duration=4456953750, load_duration=136686083, prompt_eval_count=536, prompt_eval_duration=1343776000, eval_count=133, eval_duration=2973495000, message=Message(role='assistant', content='{\n  "gameplay": {\n    "score": -1.0,\n    "quote": "The combat is totally broken and unresponsive, making it a nightmare to play."\n  },\n  "graphics": {\n    "score": null,\n    "quote": null\n  },\n  "story": {\n    "score": 0.5,\n    "quote": "the story is enjoyable enough, if a bit simple"\n  },\n  "performance": {\n    "score": null,\n    "quote": null\n  },\n  "overall": {\n    "score": -1.0,\n    "quote": null\n  }\n}', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

That is quite hard to read. We can use `.model_dump_json()` to print it nicely.

In [8]:
print(response.model_dump_json(indent=4))

{
    "model": "llama3.2",
    "created_at": "2026-08-06T19:15:04.100022Z",
    "done": true,
    "done_reason": "stop",
    "total_duration": 4456953750,
    "load_duration": 136686083,
    "prompt_eval_count": 536,
    "prompt_eval_duration": 1343776000,
    "eval_count": 133,
    "eval_duration": 2973495000,
    "message": {
        "role": "assistant",
        "content": "{\n  \"gameplay\": {\n    \"score\": -1.0,\n    \"quote\": \"The combat is totally broken and unresponsive, making it a nightmare to play.\"\n  },\n  \"graphics\": {\n    \"score\": null,\n    \"quote\": null\n  },\n  \"story\": {\n    \"score\": 0.5,\n    \"quote\": \"the story is enjoyable enough, if a bit simple\"\n  },\n  \"performance\": {\n    \"score\": null,\n    \"quote\": null\n  },\n  \"overall\": {\n    \"score\": -1.0,\n    \"quote\": null\n  }\n}",
        "thinking": null,
        "images": null,
        "tool_name": null,
        "tool_calls": null
    },
    "logprobs": null
}


### 3.2. Understanding the Metadata

The model returns more than just text. It gives us performance data that is crucial for calculating costs and latency.

* **`total_duration`**: The time taken in nanoseconds. If this is high (e.g., 10 seconds) and you have 10,000 reviews, your script could take days to run. We need to track this.
* **`eval_count`**: The number of tokens (words/parts of words) in the output. This is often how API providers bill you.
* **`message`**: The container holding the actual response.

Let's extract just the content we care about, the analysis.

In [9]:
response['message']['content']

'{\n  "gameplay": {\n    "score": -1.0,\n    "quote": "The combat is totally broken and unresponsive, making it a nightmare to play."\n  },\n  "graphics": {\n    "score": null,\n    "quote": null\n  },\n  "story": {\n    "score": 0.5,\n    "quote": "the story is enjoyable enough, if a bit simple"\n  },\n  "performance": {\n    "score": null,\n    "quote": null\n  },\n  "overall": {\n    "score": -1.0,\n    "quote": null\n  }\n}'

Let's print it to make it readable.

In [10]:
print(response['message']['content'])

{
  "gameplay": {
    "score": -1.0,
    "quote": "The combat is totally broken and unresponsive, making it a nightmare to play."
  },
  "graphics": {
    "score": null,
    "quote": null
  },
  "story": {
    "score": 0.5,
    "quote": "the story is enjoyable enough, if a bit simple"
  },
  "performance": {
    "score": null,
    "quote": null
  },
  "overall": {
    "score": -1.0,
    "quote": null
  }
}


### 3.3. Converting Strings to Dictionaries

There is a catch. The model returns a **String** that *looks* like JSON, but Python still sees it as a plain block of text. We can't access the data yet.

If we try to ask for `['gameplay']`, Python will give us an error because you can't ask for a key from a string.

In [ ]:
response['message']['content']['gameplay']

To fix this, we need to **parse** the string. This converts the text representation of the data into a real Python Dictionary.

In [11]:
# Convert the string to a validated Pydantic object
validated_data = ReviewAnalysis.model_validate_json(response['message']['content'])

# Convert the Pydantic object into a dictionary for Pandas
data = validated_data.model_dump()

data

{'gameplay': {'score': -1.0,
  'quote': 'The combat is totally broken and unresponsive, making it a nightmare to play.'},
 'graphics': {'score': None, 'quote': None},
 'story': {'score': 0.5,
  'quote': 'the story is enjoyable enough, if a bit simple'},
 'performance': {'score': None, 'quote': None},
 'overall': {'score': -1.0, 'quote': None}}

In [12]:
data['gameplay']

{'score': -1.0,
 'quote': 'The combat is totally broken and unresponsive, making it a nightmare to play.'}

In [13]:
data['gameplay']['score']

-1.0

Finally, to get this ready for our analysis, we can flatten this dictionary into a pandas DataFrame using `pd.json_normalize()`.



In [14]:
pd.json_normalize(data, sep='_')

,gameplay_score,gameplay_quote,graphics_score,graphics_quote,story_score,story_quote,performance_score,performance_quote,overall_score,overall_quote
0,-1.0,"The combat is totally broken and unresponsive,...",None,None,0.5,"the story is enjoyable enough, if a bit simple",None,None,-1.0,None


The `sep='_'` tells pandas to combine nested keys with an underscore (e.g., `gameplay` + `score` becomes `gameplay_score`).

---
## 4.&nbsp; Processing a DataFrame 🧱

Now that we have the logic for one review, we need to scale this up. We'll define functions to handle the heavy lifting for us.

**1. The Analysis Function**

This function wraps our `ollama.chat` call in a `try...except` block. This ensures that if one review fails (maybe the model hiccups), our whole script doesn't crash.

In [15]:
def analyse_review_local(review_text, review_id, retries=3):
    """
    Gets the raw ollama response. Retries on failure.
    Returns None only after all retries fail.
    """
    for attempt in range(retries):
        try:
            response = ollama.chat(
                model='llama3.2',
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': review_text}
                ],
                format=ReviewAnalysis.model_json_schema(),
                options={'temperature': 0}
            )

            # Convert the ChatResponse object to a standard dictionary
            if hasattr(response, 'model_dump'):
                return response.model_dump()
            else:
                return dict(response)

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < retries - 1:
                time.sleep(1)
            else:
                print(f"All {retries} attempts failed for review {review_id}.")
                return None

**2. The Metadata Parser**

This function extracts the technical details (duration, token counts) so we can monitor performance.

In [16]:
# Define the expected structure for our technical metadata
class OllamaMetadata(BaseModel):
    # We map the raw Ollama keys to our preferred DataFrame column names
    model_name: str | None = Field(default=None, alias="model")
    created_at: str | None = None
    total_duration_ns: int | None = Field(default=None, alias="total_duration")
    load_duration_ns: int | None = Field(default=None, alias="load_duration")
    prompt_eval_duration_ns: int | None = Field(default=None, alias="prompt_eval_duration")
    generation_duration_ns: int | None = Field(default=None, alias="eval_duration")
    input_token_count: int | None = Field(default=None, alias="prompt_eval_count")
    output_token_count: int | None = Field(default=None, alias="eval_count")

`generation_duration_ns: int | None = Field(default=None, alias="eval_duration")` tells Pydantic: "When you look at the raw Ollama dictionary, look for the key `eval_duration`, but when you output the data for pandas, rename it to `generation_duration_ns`."

In [17]:
def parse_metadata(response, review_id):
    """
    Extracts technical metadata using Pydantic.
    Returns a structure with None values if keys are missing.
    """
    # 1. Instantly create an empty fallback structure
    fallback_data = OllamaMetadata().model_dump()
    fallback_data["review_id"] = review_id

    if response is None:
        return fallback_data

    try:
        # 2. Validate the dictionary returned by Ollama.
        # Pydantic automatically handles the aliases (renaming keys)!
        validated_meta = OllamaMetadata.model_validate(response)

        # 3. Convert back to dict and append the ID
        final_dict = validated_meta.model_dump()
        final_dict["review_id"] = review_id

        return final_dict

    except Exception as e:
        print(f"Error parsing metadata for {review_id}: {e}")
        return fallback_data

**3. The Sentiment Parser**

This function takes the raw JSON string from the model and converts it into a Python dictionary. Crucially, it handles errors: if the model returns bad JSON (which can happen!), it returns a "default structure" full of `None` values instead of crashing.

In [18]:
def parse_sentiment(response, review_id):
    """
    Extracts the aspect scores using Pydantic validation.
    Returns a consistent structure with None values if parsing fails.
    """
    # 1. Let Pydantic instantly create our empty fallback structure
    fallback_data = ReviewAnalysis().model_dump()
    fallback_data["review_id"] = review_id

    if response is None:
        return fallback_data

    try:
        message = response.get('message', {})
        content_str = message.get('content', '{}')

        # 2. Validate the JSON string directly against our Pydantic model
        validated_data = ReviewAnalysis.model_validate_json(content_str)

        # 3. Convert back to dict and append the ID
        final_dict = validated_data.model_dump()
        final_dict["review_id"] = review_id

        return final_dict

    except Exception as e:
        # Pydantic will raise a ValidationError if the output is malformed
        print(f"Error parsing sentiment for review {review_id}: {e}")
        return fallback_data

**4. The Main Loop**

Finally, we bring it all together. This function loops through every row in your DataFrame, sends the text to the model, parses the result, and collects the data into two tidy DataFrames: one for sentiment scores and one for technical metadata.

In [19]:
def process_game_reviews(df):
    sentiment_records = []
    metadata_records = []

    for index, row in df.iterrows():
        r_id = row['review_id']
        text = row['review_text']

        # 1. Get raw response
        raw_response = analyse_review_local(text, r_id)

        # 2. Extract Data
        sentiment_data = parse_sentiment(raw_response, r_id)
        meta_data = parse_metadata(raw_response, r_id)

        sentiment_records.append(sentiment_data)
        metadata_records.append(meta_data)

    # --- Create DataFrames ---
    df_aspects = pd.json_normalize(sentiment_records, sep='_')
    if not df_aspects.empty:
        df_aspects = df_aspects.set_index('review_id')

    df_metadata = pd.DataFrame(metadata_records)
    if not df_metadata.empty:
        df_metadata = df_metadata.set_index('review_id')

    return df_aspects, df_metadata

### 4.1. Running on Dummy Data

Let's create a small dummy dataset to verify our pipeline works correctly.

In [20]:
data = [
    {
        "review_id": 101,
        "game_title": "Star  Drifter",
        "review_text": "Absolutely stunning visuals and the soundtrack is a masterpiece. The combat feels fluid and responsive. I haven't put the controller down in three days!"
    },
    {
        "review_id": 102,
        "game_title": "Dungeon Crawler X",
        "review_text": "I really wanted to like this, but the bugs make it unplayable. It crashes every time I open the inventory. Waste of money until they patch it."
    },
    {
        "review_id": 103,
        "game_title": "Cyber Racer 2099",
        "review_text": "The driving physics are incredible, easily the best in the series. However, the story mode is boring and the voice acting is pretty cringe. Mixed bag."
    },
    {
        "review_id": 104,
        "game_title": "Farm Sim Ultra",
        "review_text": "It's a farming simulator. You plant corn and wait. Game works fine."
    },
    {
        "review_id": 105,
        "game_title": "Knights of Valor",
        "review_text": "Great character customisation and lore. The UI is a disaster though—text is too small to read on a TV and the menus are confusing."
    }
]

df = pd.DataFrame(data)

df

,review_id,game_title,review_text
0,101,Star Drifter,Absolutely stunning visuals and the soundtrack...
1,102,Dungeon Crawler X,"I really wanted to like this, but the bugs mak..."
2,103,Cyber Racer 2099,"The driving physics are incredible, easily the..."
3,104,Farm Sim Ultra,It's a farming simulator. You plant corn and w...
4,105,Knights of Valor,Great character customisation and lore. The UI...


In [21]:
ollama_aspects, ollama_metadata = process_game_reviews(df)

Now let's verify the results. We should see two distinct dataframes.

In [22]:
ollama_aspects

,gameplay_score,gameplay_quote,graphics_score,graphics_quote,story_score,story_quote,performance_score,performance_quote,overall_score,overall_quote
review_id,,,,,,,,,,
101,1.0,I haven't put the controller down in three days!,1.0,NaN,NaN,None,NaN,None,1.0,NaN
102,-1.0,Waste of money until they patch it.,NaN,NaN,NaN,None,-1.0,None,-1.0,NaN
103,1.0,"The driving physics are incredible, easily the...",NaN,NaN,-0.5,None,NaN,None,-0.5,Mixed bag.
104,0.0,NaN,0.0,NaN,0.0,None,0.0,None,0.0,NaN
105,NaN,character customisation and lore.,-1.0,text is too small to read on a TV,NaN,None,NaN,None,-0.5,The UI is a disaster though—text is too small ...


In [23]:
ollama_metadata

,model_name,created_at,total_duration_ns,load_duration_ns,prompt_eval_duration_ns,generation_duration_ns,input_token_count,output_token_count
review_id,,,,,,,,
101,llama3.2,2026-08-06T20:03:11.996192Z,6199344959,2225799959,1332021000,2636050000,522,118
102,llama3.2,2026-08-06T20:03:14.871382Z,2869257458,98806583,179098000,2589674000,527,116
103,llama3.2,2026-08-06T20:03:17.908465Z,3032224333,99599000,179047000,2751329000,526,122
104,llama3.2,2026-08-06T20:03:20.666074Z,2752562167,100489542,101008000,2549610000,510,114
105,llama3.2,2026-08-06T20:03:24.104487Z,3433635708,101103583,181813000,3149156000,523,140


Finally, let's save these to a CSV.

In [ ]:
ollama_aspects.to_csv('../data/ollama_aspects.csv', index=False)
ollama_metadata.to_csv('../data/ollama_metadata.csv', index=False)

---
## 5.&nbsp; The Challenge: Processing Your Data 🚀

Now it's time to run this on the dataset you labelled in the previous notebook. Here is your mission:

1.  **Load your data:** Import the `labeled_reviews.csv` file you downloaded in the previous notebook.
2.  **Sample the data:** Running hundreds of rows locally will be slow. Create a new dataframe called `df_sample` with just 10 rows to start with.
3.  **Create you functions:** One by one, create the building blocks of your analyser. Begin with the LLM, then extract and structure the metadata. Then extract and structure the Aspect based ANalysis. Then bring it all togeher and loop through your sample dataframe.
4. **Read the result:** Once it's working on your sample dataframe, investigate the outputs. Are you happy with them? Are there any you disagree with? Revise your system prompt and try again. Keep iterating until you're happy with the output.
5. **Scale up:** Once you're happy with your sample output, run the analysis on your whole dataframe.
6. **Don't lose your work:** Don't forget to save your new dataframes as CSVs.
7. **Time is precious:** Make sure to write down somewhere how long it took to run through all 200 reviews, this will help you analysis later.

---
## Key Takeaways 🧠

* **Strict Instructions:** LLMs need to be told *exactly* what format to output, or they will ramble.
* **Structured Data:** Using `format='json'` allows us to integrate AI into standard data pipelines.
* **Local Cost:** Running this was free in terms of money, but did you notice the speed? If you had 10,000 reviews, how long would this take? Keep that number in mind for your final report.